In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/train.csv")
print(f"Loaded from CSV: {df.shape}")


df = df.rename(
    columns={
        "query": "query_text",
        "region_res5": "user_region_res5",
        "region_res4": "user_region_res4",
    }
).copy()
df["sample_weight"] = np.log1p(df["total_orders"])

df.query("query_text == 'massage'").head(20)


Loaded from CSV: (14521, 9)


,query_text,user_region_res5,user_region_res4,label,total_orders,source,share_near,share_far,dist_segment,sample_weight
5,massage,85283297fffffff,8428329ffffffff,0,50,REAL,1.0,0.0,LOCAL,3.931826
15,massage,852a104ffffffff,842a105ffffffff,0,50,REAL,1.0,0.0,LOCAL,3.931826
133,massage,8526de7bfffffff,8426de7ffffffff,0,50,REAL,0.8,0.2,LOCAL,3.931826
265,massage,85274da7fffffff,84274dbffffffff,0,50,REAL,1.0,0.0,LOCAL,3.931826
308,massage,85441e0bfffffff,84441e1ffffffff,0,50,REAL,0.8,0.0,LOCAL,3.931826
447,massage,8529a413fffffff,8429a41ffffffff,0,50,REAL,0.8,0.2,LOCAL,3.931826
449,massage,8548d897fffffff,8448d89ffffffff,0,50,REAL,1.0,0.0,LOCAL,3.931826
450,massage,852a8473fffffff,842a847ffffffff,0,50,REAL,1.0,0.0,LOCAL,3.931826
476,massage,8528d593fffffff,8428d59ffffffff,0,50,REAL,0.8,0.2,LOCAL,3.931826
558,massage,852ab337fffffff,842ab33ffffffff,0,50,REAL,1.0,0.0,LOCAL,3.931826


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, mean_absolute_error
from sklearn.base import BaseEstimator, TransformerMixin
from sentence_transformers import SentenceTransformer

class SentenceTransformerWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="all-MiniLM-L6-v2", device="cpu"):
        self.model_name = model_name
        self.device = device
        self.model = None
    
    def fit(self, X, y=None):
        if self.model is None:
            self.model = SentenceTransformer(self.model_name, device=self.device)
        return self
    
    def transform(self, X):
        if isinstance(X, pd.Series):
            texts = X.tolist()
        else:
            texts = list(X)
        embeddings = self.model.encode(texts, show_progress_bar=False)
        return embeddings


class ModelWrapper:
    def __init__(self, pipeline_model):
        self.pipeline = pipeline_model
        # Match renamed columns
        self.required_columns = ["query_text", "user_region_res5", "user_region_res4"]
    
    def _prepare_input(self, data):
        if isinstance(data, dict):
            row = {
                "query_text": str(data.get("query_text", "")),
                "user_region_res5": data.get("user_region_res5") or "UNKNOWN",
                "user_region_res4": data.get("user_region_res4") or "UNKNOWN",
            }
            return pd.DataFrame([row])
        elif isinstance(data, list):
            rows = []
            for item in data:
                if isinstance(item, dict):
                    row = {
                        "query_text": str(item.get("query_text", "")),
                        "user_region_res5": item.get("user_region_res5") or "UNKNOWN",
                        "user_region_res4": item.get("user_region_res4") or "UNKNOWN",
                    }
                    rows.append(row)
                else:
                    raise ValueError("List items must be dictionaries")
            return pd.DataFrame(rows)
        elif isinstance(data, pd.DataFrame):
            return data[self.required_columns].copy()
        else:
            raise ValueError("Input must be dict, list of dicts, or DataFrame")
    
    def predict(self, data):
        X = self._prepare_input(data)
        return self.pipeline.predict(X)
    
    def predict_proba(self, data):
        X = self._prepare_input(data)
        return self.pipeline.predict_proba(X)
    
    def predict_single(self, query_text, user_region_res5=None, user_region_res4=None):
        data = {
            "query_text": query_text,
            "user_region_res5": user_region_res5,
            "user_region_res4": user_region_res4,
        }
        pred_segment = int(self.predict(data)[0])
        prob_vec = self.predict_proba(data)[0]
        confidence = float(prob_vec[pred_segment])
        return {"predicted_segment": pred_segment, "confidence": confidence}





In [3]:
# Features and target from the renamed df
X = df[["query_text", "user_region_res5", "user_region_res4"]]
y = df["label"].astype(int)
w = df["sample_weight"] #= np.log1p(df["total_orders"])

w.head()

0    3.931826
1    3.931826
2    3.931826
3    3.931826
4    3.931826
Name: sample_weight, dtype: float64

In [4]:
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X,
    y,
    w,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

text_features = "query_text"
cat_features = ["user_region_res5", "user_region_res4"]

preprocess = ColumnTransformer(
    transformers=[
        ("text", SentenceTransformerWrapper(model_name="all-MiniLM-L6-v2"), text_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ],
    remainder="drop",
)

model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            class_weight=None,
            multi_class="multinomial",
        )),
    ]
)

print("Training (4 classes)...")
model.fit(X_train, y_train, clf__sample_weight=w_train)

y_pred = model.predict(X_val)
print("\nValidation (4 classes):")
print(classification_report(y_val, y_pred))

mae_buckets = mean_absolute_error(y_val, y_pred)
print(f"\nMAE on 4-class bucket_id (0..3): {mae_buckets:.3f}")

wrapped_model = ModelWrapper(model)

Training (4 classes)...


/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Users/zphilipp/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZ


Validation (4 classes):
              precision    recall  f1-score   support

           0       0.61      0.73      0.66       986
           1       0.49      0.36      0.42       496
           2       0.41      0.26      0.32       267
           3       0.73      0.76      0.74      1156

    accuracy                           0.63      2905
   macro avg       0.56      0.53      0.53      2905
weighted avg       0.62      0.63      0.62      2905


MAE on 4-class bucket_id (0..3): 0.682


In [5]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_pred, labels=[0,1,2,3])
print(cm)

[[715  99  24 148]
 [192 181  34  89]
 [ 74  42  69  82]
 [190  50  43 873]]


In [6]:
import joblib

MODEL_PATH = "query_region_radius_model_st.joblib"
# Save the wrapped model (which contains the pipeline)
joblib.dump(wrapped_model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

# Also save the raw pipeline for backward compatibility if needed
PIPELINE_PATH = "query_region_radius_pipeline_st.joblib"
joblib.dump(model, PIPELINE_PATH)
print(f"Pipeline saved to {PIPELINE_PATH}")

Model saved to query_region_radius_model_st.joblib
Pipeline saved to query_region_radius_pipeline_st.joblib


In [7]:
def predict_radius_segment(query_text, user_region_res5=None, user_region_res4=None):
    """
    Predict radius segment for a query.
    
    Args:
        query_text: The search query text
        user_region_res5: H3 region at resolution 5 (optional)
        user_region_res4: H3 region at resolution 4 (optional)
    
    Returns:
        dict with 'predicted_segment' and 'confidence'
    """
    #return wrapped_model.predict_single(query_text, user_region_res5, user_region_res4)
    return wrapped_model.predict_single(query_text, user_region_res5, user_region_res4)


In [10]:
import pandas as pd

# Test queries - same as in the original notebook
all_rows = [
    # --- test_rows ---
    {"query_text": "universal",        "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "massage",          "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "masage",           "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "massage",          "user_region_res5": "8544f003fffffff", "user_region_res4": "8444f01ffffffff"},
    {"query_text": "facial",           "user_region_res5": "852b9bc7fffffff", "user_region_res4": "842b9bdffffffff"},
    {"query_text": "smog check",       "user_region_res5": "8528308bfffffff", "user_region_res4": "8428309ffffffff"},
    {"query_text": "oil change",       "user_region_res5": "85269607fffffff", "user_region_res4": "8426961ffffffff"},
    {"query_text": "trampoline park",  "user_region_res5": "852a1073fffffff", "user_region_res4": "842a107ffffffff"},
    {"query_text": "escape room",      "user_region_res5": "8526640ffffffff", "user_region_res4": "8426641ffffffff"},
    {"query_text": "zoo",              "user_region_res5": "8529a54ffffffff", "user_region_res4": "8429a55ffffffff"},
    {"query_text": "sky zone",         "user_region_res5": "8526640ffffffff", "user_region_res4": "8426641ffffffff"},
    {"query_text": "bowling",          "user_region_res5": "85262cd7fffffff", "user_region_res4": "84262cdffffffff"},
    {"query_text": "seaworld",         "user_region_res5": "852986bbfffffff", "user_region_res4": "842986bffffffff"},
    {"query_text": "great wolf lodge", "user_region_res5": "85283471fffffff", "user_region_res4": "8428347ffffffff"},
    {"query_text": "great wolf",       "user_region_res5": "85283471fffffff", "user_region_res4": "8428347ffffffff"},
    {"query_text": "citypass",         "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "whale watching",   "user_region_res5": "8529a54ffffffff", "user_region_res4": "8429a55ffffffff"},
    {"query_text": "hotel",            "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "hotl",             "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "windows 11",       "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "microsoft office", "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "costco",           "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "cosco",            "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "windows 10",       "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "airport parking", "user_region_res5": "8529b6d3fffffff", "user_region_res4": "8429b6dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664cffffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664c3fffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "bared monkey", "user_region_res5": "852a1073fffffff", "user_region_res4": "842a107ffffffff"},
    {"query_text": "bible museum", "user_region_res5": "852aa847fffffff", "user_region_res4": "842aa85ffffffff"},
    {"query_text": "big air", "user_region_res5": "8544da87fffffff", "user_region_res4": "8444da9ffffffff"},
    {"query_text": "big air trampoline park", "user_region_res5": "8544d84ffffffff", "user_region_res4": "8444d85ffffffff"},
    {"query_text": "laser hair removal", "user_region_res5": "852664c3fffffff", "user_region_res4": "842664dffffffff"},
]

# Test predictions using wrapped model (works with list of dicts, no DataFrame needed)
pred_segments = wrapped_model.predict(all_rows)
probs = wrapped_model.predict_proba(all_rows)

# Create output DataFrame for display
df_out = pd.DataFrame(all_rows)
df_out["predicted_segment"] = pred_segments.astype(int)
df_out["confidence"] = [float(probs[i][pred_segments[i]]) for i in range(len(pred_segments))]

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

df_out


,query_text,user_region_res5,user_region_res4,predicted_segment,confidence
0,universal,852a1423fffffff,842a143ffffffff,3,0.837304
1,massage,852a1423fffffff,842a143ffffffff,0,0.420664
2,masage,852a1423fffffff,842a143ffffffff,0,0.451589
3,massage,8544f003fffffff,8444f01ffffffff,3,0.450487
4,facial,852b9bc7fffffff,842b9bdffffffff,0,0.800553
5,smog check,8528308bfffffff,8428309ffffffff,0,0.843708
6,oil change,85269607fffffff,8426961ffffffff,0,0.450190
7,trampoline park,852a1073fffffff,842a107ffffffff,0,0.659123
8,escape room,8526640ffffffff,8426641ffffffff,1,0.811880
9,zoo,8529a54ffffffff,8429a55ffffffff,1,0.372425


In [9]:
# Example: Loading and using the model independently (no DataFrame dependency)
import joblib

# Load the wrapped model
loaded_model = joblib.load("query_region_radius_model_st.joblib")

# Use with dict (single prediction)
result1 = loaded_model.predict_single("massage", "852a1423fffffff", "842a143ffffffff")
print("Single prediction (dict input):", result1)

# Use with list of dicts (batch prediction)
test_data = [
    {"query_text": "massage", "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "hotel", "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
]
results = loaded_model.predict(test_data)
print("\nBatch predictions:", results)

# The model works without any DataFrame structure being stored or required


Single prediction (dict input): {'predicted_segment': 0, 'confidence': 0.42066436999140167}

Batch predictions: [0 3]
